<a href="https://colab.research.google.com/github/deepshresthaa/A-Clustered-Graph-Based-Framework-for-Semantic-Research-Paper-Retrieval-and-Recommendation/blob/main/code/05_hyperparameter_tuning_for_localized_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torch-geometric optuna
import os
import torch
import numpy as np
import networkx as nx
import optuna

output_dir = "/content/drive/MyDrive/gnn_cluster_models"
optuna.logging.set_verbosity(optuna.logging.WARNING)

def optimize_pq_parameters(target_cluster_id, query_vector, n_trials=20, max_depth=3):
    """
    Loads a specific cluster checkpoint, builds its NetworkX graph,
    and runs Optuna to find the optimal p and q traversal parameters.
    """
    checkpoint_path = os.path.join(output_dir, f"cluster_{target_cluster_id}.pt")
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint for cluster {target_cluster_id} not found at {checkpoint_path}")

    # 1. Load the cluster model file
    cluster_data = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    graph_data = cluster_data['graph_data']

    # 2. Compute cosine similarities to find the seed node
    query_t = torch.as_tensor(query_vector, dtype=torch.float32).unsqueeze(0)
    node_sims = torch.cosine_similarity(query_t, graph_data.x, dim=1).detach().numpy()
    seed_idx = int(np.argmax(node_sims))

    # 3. Build NetworkX graph from saved edges
    G = nx.Graph()
    edges = graph_data.edge_index.numpy()
    for u, v in zip(edges[0], edges[1]):
        G.add_edge(u, v)

    # 4. Optuna Objective Function for p and q optimization
    def objective(trial):
        p = trial.suggest_float("p", 0.1, 5.0)
        q = trial.suggest_float("q", 0.1, 5.0)

        visited_scores = {}
        queue = [(seed_idx, 0, 1.0)]

        while queue:
            curr_node, depth, prob = queue.pop(0)
            if curr_node in visited_scores or depth > max_depth:
                continue

            combined_score = 0.6 * float(node_sims[curr_node]) + 0.4 * prob
            visited_scores[curr_node] = combined_score

            neighbors = list(G.neighbors(curr_node)) if G.has_node(curr_node) else []
            for nbr in neighbors:
                bias = (1 / p) if nbr == seed_idx else (1 / q)
                next_prob = prob * (1.0 / len(neighbors)) * bias
                queue.append((nbr, depth + 1, next_prob))

        if not visited_scores:
            return float('inf')

        # We want to maximize the average path relevance score,
        # so we return the negative value for Optuna's minimization engine.
        return -np.mean(list(visited_scores.values()))

    # 5. Run the Optuna Study
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    best_p = study.best_params["p"]
    best_q = study.best_params["q"]
    best_score = -study.best_value

    return {"p": best_p, "q": best_q}, best_score

# Pick a target cluster and a query vector (e.g., sample embedding from the cluster file)
sample_cluster_id = 12
sample_ckpt = torch.load(os.path.join(output_dir, f"cluster_{sample_cluster_id}.pt"), map_location='cpu', weights_only=False)
dummy_query = sample_ckpt['graph_data'].x[0].numpy()

best_params, evaluation_score = optimize_pq_parameters(
    target_cluster_id=sample_cluster_id,
    query_vector=dummy_query,
    n_trials=15
)

print(f"Optimal Parameters Found for Cluster {sample_cluster_id}:")
print(f"  -> p: {best_params['p']:.4f}")
print(f"  -> q: {best_params['q']:.4f}")
print(f"  -> Best Path Score: {evaluation_score:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Optimal Parameters Found for Cluster 12:
  -> p: 0.7442
  -> q: 2.9471
  -> Best Path Score: 1.0000
